# 3_4 — Object-held-out CV с доверительными интервалами (P0-a)

**Primary:** balanced **repeated** stratified grouped K-fold по объектам. Объекты распределяются с гарантией **CRR-объекта в каждом тест-фолде** и val из ≥2 объектов **обоих классов** (исправлены замечания о слабой стратификации/валидации). **Secondary:** LOO по объектам.

Неопределённость DPI-Flow на инференсе **пропагируется через conditional flow** (MC, `config.mc_samples_eval` сэмплов θ), а не берётся posterior mean.

> ⚠️ Сначала ПЕРЕматериализуйте `data/dataset` (ноутбуки 1_x, `prefix_strict_preonset=True`) и переобучите модели, иначе headline-числа невалидны (утечка префикса) и устаревший `config.json`.

Корректные CI считаются в 3_5 (**object-cluster bootstrap по pooled OOF**), не наивным √n_folds.

In [1]:
import os, sys, json, time
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if not os.path.exists(os.path.join(REPO, 'src')): REPO = os.getcwd()
sys.path.insert(0, os.path.join(REPO, 'src')); os.chdir(REPO)
import numpy as np, pandas as pd
TABLES=os.path.join(REPO,'results','analysis_tables'); os.makedirs(TABLES, exist_ok=True)
PUB=os.path.join(REPO,'results','tables'); os.makedirs(PUB, exist_ok=True)
FIGS=os.path.join(REPO,'results','analysis_figs'); os.makedirs(FIGS, exist_ok=True)
print('REPO =', REPO)

REPO = /Users/nikita/Desktop/projects/liquefaction-ai


In [2]:
from liquefaction_ai import load_population_artifact
from liquefaction_ai.evaluation.cross_validation import build_folds, evaluate_fold, aggregate_cv
import torch
device = torch.device('cpu')
QUICK = False        # True = demo-эпохи (дымовой тест)
N_SPLITS = 5
N_REPEATS = 3        # repeated CV (≥3) — требование рецензента; >1 усредняет повторы
pop, config = load_population_artifact('data/dataset')
meta = pop['meta']
# idempotency-штамп протокола (перезапуск перезаписывает, дублей нет)
meta_stamp = {'prefix_strict_preonset': getattr(config,'prefix_strict_preonset',None),
              'seed': config.seed, 'n_splits': N_SPLITS, 'n_repeats': N_REPEATS,
              'mc_samples_eval': getattr(config,'mc_samples_eval',1), 'ts': time.strftime('%Y-%m-%d %H:%M')}
json.dump(meta_stamp, open(os.path.join(TABLES,'cv_grouped_run_meta.json'),'w'), ensure_ascii=False, indent=2)
print('samples', len(meta), '| objects', meta['object'].nunique(), '|', meta_stamp)

samples 1093 | objects 20 | {'prefix_strict_preonset': True, 'seed': 42, 'n_splits': 5, 'n_repeats': 3, 'mc_samples_eval': 8, 'ts': '2026-06-29 13:34'}


## Primary: balanced repeated grouped CV (все сильные baselines)

In [3]:
folds = build_folds(meta, config, seed=42, loo=False, n_splits=N_SPLITS, n_repeats=N_REPEATS)
raw, samples = [], []
for gid, fold in enumerate(folds):
    r, s = evaluate_fold(pop, config, fold, gid, device, quick=QUICK)
    raw.append(r); samples.append(s)
    print(f"rep{fold['repeat']} fold{fold['fold']}: ", dict(zip(r.model, r.P3_Core.round(1))))
raw = pd.concat(raw, ignore_index=True); samples = pd.concat(samples, ignore_index=True)
raw.to_csv(os.path.join(TABLES,'cv_grouped_raw.csv'), index=False)        # overwrite (идемпотентно)
samples.to_csv(os.path.join(TABLES,'cv_grouped_samples.csv'), index=False)
summary = aggregate_cv(raw); summary.to_csv(os.path.join(TABLES,'cv_grouped_summary.csv'), index=False)
print('models in CV:', sorted(raw.model.unique()))
summary[[c for c in summary.columns if c in ('model','n_folds') or c.endswith('_mean')][:12]]

rep0 fold0:  {'DPI-Flow': 160.4, 'DPI-EVT': 204.8, 'EVT-NeuralSSM': 189.7, 'PINN': 100.0, 'Transformer': 140.1, 'Neural Spline Flow': 112.2, 'GRU': 102.2, 'TCN': 79.7, 'CatBoost': nan}
rep0 fold1:  {'DPI-Flow': 250.8, 'DPI-EVT': 301.7, 'EVT-NeuralSSM': 304.1, 'PINN': 100.0, 'Transformer': 222.6, 'Neural Spline Flow': 165.7, 'GRU': 121.8, 'TCN': 174.7, 'CatBoost': nan}
rep0 fold2:  {'DPI-Flow': 306.1, 'DPI-EVT': 400.3, 'EVT-NeuralSSM': 404.8, 'PINN': 100.0, 'Transformer': 436.1, 'Neural Spline Flow': 270.5, 'GRU': 269.1, 'TCN': 116.1, 'CatBoost': nan}
rep0 fold3:  {'DPI-Flow': 303.7, 'DPI-EVT': 273.9, 'EVT-NeuralSSM': 257.5, 'PINN': 100.0, 'Transformer': 207.9, 'Neural Spline Flow': 146.5, 'GRU': 127.1, 'TCN': 83.5, 'CatBoost': nan}
rep0 fold4:  {'DPI-Flow': 197.4, 'DPI-EVT': 159.9, 'EVT-NeuralSSM': 204.1, 'PINN': 100.0, 'Transformer': 137.3, 'Neural Spline Flow': 105.7, 'GRU': 76.2, 'TCN': 44.3, 'CatBoost': nan}
rep1 fold0:  {'DPI-Flow': 216.7, 'DPI-EVT': 216.8, 'EVT-NeuralSSM': 208.3,

KeyboardInterrupt: 

> Наивный `*_ci95` в summary — только справочная дисперсия по фолдам. **Публикационные CI** — в 3_5 (object-cluster bootstrap).

## Secondary: leave-one-object-out (20 фолдов, тяжелее)

In [ ]:
RUN_LOO = True
if RUN_LOO:
    lf = build_folds(meta, config, seed=42, loo=True)
    rl, sl = [], []
    for gid, fold in enumerate(lf):
        r, s = evaluate_fold(pop, config, fold, gid, device, quick=QUICK); rl.append(r); sl.append(s)
    pd.concat(rl, ignore_index=True).to_csv(os.path.join(TABLES,'cv_loo_raw.csv'), index=False)
    pd.concat(sl, ignore_index=True).to_csv(os.path.join(TABLES,'cv_loo_samples.csv'), index=False)
    print('LOO done')
else: print('LOO пропущен')